# LungInsight — Annotation-Free Detection Validation

Runs model inference on candidates detected locally without annotations,
then validates predictions against pylidc ground truth.

**Before running this notebook:**
1. Run `detect_candidates_cpu.py` locally to produce `candidate_patches/`
2. Upload the entire `candidate_patches/` folder to `MyDrive/LungInsight/`
3. Upload `cir_multihead_pipeline.py` and `se_resnet3d.py` to `/content/LungInsight/`
4. Enable GPU runtime (Runtime → Change runtime type → T4 GPU)

In [ ]:
%pip install -q torch torchvision numpy pandas scipy grad-cam pylidc

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

ROOT_DIR = '/content/LungInsight'
if not os.path.isdir(ROOT_DIR):
    raise RuntimeError('Upload cir_multihead_pipeline.py and se_resnet3d.py to /content/LungInsight')
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

from cir_multihead_pipeline import create_multihead_model, FEATURE_NAMES, generate_characteristic_heatmaps

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Config — set paths here

In [ ]:
DRIVE_DIR      = '/content/drive/MyDrive/LungInsight'
CANDIDATES_DIR = f'{DRIVE_DIR}/candidate_patches'   # uploaded folder
CHECKPOINT     = f'{DRIVE_DIR}/best_model_gpu.pth'
MANIFEST_PATH  = f'{CANDIDATES_DIR}/candidates_manifest.csv'

# Set to a specific patient_id to only run that scan, or None for all
FILTER_PATIENT_ID = None   # e.g. 'LIDC-IDRI-0001'

# Whether to generate Grad-CAM heatmaps for TP candidates (slower)
GENERATE_HEATMAPS = True

for label, path in [('Manifest', MANIFEST_PATH), ('Checkpoint', CHECKPOINT)]:
    ok = os.path.isfile(path)
    print(f'{label}: {path}  [{"OK" if ok else "NOT FOUND"}]')
    if not ok:
        raise FileNotFoundError(path)

## Load model and manifest

In [ ]:
model = create_multihead_model(head_names=FEATURE_NAMES, device=device)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()
print('Model loaded.')

manifest = pd.read_csv(MANIFEST_PATH)

# Rewrite local Windows paths to Drive paths
manifest['file_path'] = manifest['file_path'].apply(
    lambda p: os.path.join(CANDIDATES_DIR, os.path.basename(str(p)))
    if isinstance(p, str) and p else ''
)

if FILTER_PATIENT_ID:
    manifest = manifest[manifest['patient_id'] == FILTER_PATIENT_ID].reset_index(drop=True)

print(f'Manifest rows: {len(manifest)}')
print(manifest['status'].value_counts().to_string())

## Run inference on all candidates (TP + FP rows)

In [ ]:
# Initialise prediction columns
for feat in FEATURE_NAMES:
    manifest[f'{feat}_pred'] = np.nan

candidate_rows = manifest[manifest['status'].isin(['TP', 'FP'])].copy()
print(f'Running inference on {len(candidate_rows)} candidates ...')

for idx, row in candidate_rows.iterrows():
    fpath = row['file_path']
    if not isinstance(fpath, str) or not os.path.isfile(fpath):
        continue
    patch = np.load(fpath).astype(np.float32)
    x = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(x)
    for feat in FEATURE_NAMES:
        manifest.at[idx, f'{feat}_pred'] = float(outputs[feat].cpu().item())

print('Inference complete.')

## Truth table and detection metrics

In [ ]:
tp = (manifest['status'] == 'TP').sum()
fp = (manifest['status'] == 'FP').sum()
fn = (manifest['status'] == 'FN').sum()
n_ann = tp + fn
n_det = tp + fp

print('=== Detection Summary ===')
print(f'  Candidates detected : {n_det}')
print(f'  Annotated nodules   : {n_ann}')
print(f'  True Positives (TP) : {tp}')
print(f'  False Positives (FP): {fp}')
print(f'  False Negatives (FN): {fn}')
print(f'  Sensitivity (recall): {tp/n_ann*100:.1f}%' if n_ann else '  Sensitivity: N/A')
print(f'  Precision           : {tp/n_det*100:.1f}%' if n_det else '  Precision: N/A')

# Regression metrics for TP pairs (predicted vs ground truth)
tp_rows = manifest[manifest['status'] == 'TP']
print(f'\n=== Regression Metrics (TP matches, n={len(tp_rows)}) ===')
header = f"  {'feature':<22}{'MAE':>8}{'RMSE':>8}{'Pearson r':>12}{'n':>6}"
print(header)
print('  ' + '-' * (len(header) - 2))
for feat in FEATURE_NAMES:
    y_pred = tp_rows[f'{feat}_pred'].values.astype(float)
    y_true = tp_rows[f'{feat}_gt'].values.astype(float)
    valid  = ~np.isnan(y_true) & ~np.isnan(y_pred)
    n = valid.sum()
    if n == 0:
        print(f'  {feat:<22}     N/A     N/A          N/A     0')
        continue
    yp, yt = y_pred[valid], y_true[valid]
    mae  = np.mean(np.abs(yp - yt))
    rmse = np.sqrt(np.mean((yp - yt) ** 2))
    r    = float(np.corrcoef(yp, yt)[0, 1]) if n > 1 and np.std(yt) > 0 else float('nan')
    print(f'  {feat:<22}{mae:>8.4f}{rmse:>8.4f}{r:>12.4f}{n:>6d}')

In [ ]:
# Display the truth table sorted by status then malignancy_pred
display_cols = (
    ['candidate_id', 'patient_id', 'status', 'diameter_mm',
     'match_distance_mm']
    + [f'{feat}_pred' for feat in FEATURE_NAMES]
    + [f'{feat}_gt'   for feat in FEATURE_NAMES]
)
display_cols = [c for c in display_cols if c in manifest.columns]
display(manifest[display_cols].sort_values(['status', 'malignancy_pred'],
                                            ascending=[True, False])
        .reset_index(drop=True)
        .style.format({c: '{:.3f}' for c in display_cols
                       if manifest[c].dtype == float})
        .background_gradient(subset=['malignancy_pred'], cmap='RdYlGn_r'))

## Save results to Drive

In [ ]:
out_csv = f'{DRIVE_DIR}/detection_validation_results.csv'
manifest.to_csv(out_csv, index=False)
print(f'Results saved: {out_csv}')

## Grad-CAM heatmaps for TP candidates

Generates one visualisation per TP match. Set `GENERATE_HEATMAPS = False`
in the config cell to skip this section.

In [ ]:
if not GENERATE_HEATMAPS:
    print('GENERATE_HEATMAPS=False, skipping.')
else:
    tp_rows = manifest[manifest['status'] == 'TP']
    for _, row in tp_rows.iterrows():
        fpath = row['file_path']
        if not isinstance(fpath, str) or not os.path.isfile(fpath):
            print(f'Patch not found, skipping: {fpath}')
            continue

        patch = np.load(fpath).astype(np.float32)
        x = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0)

        # Grad-CAM on CPU to avoid OOM
        model_cpu = model.cpu()
        heatmaps  = generate_characteristic_heatmaps(model_cpu, x, device='cpu')
        model.to(device)
        torch.cuda.empty_cache()

        mid = patch.shape[0] // 2
        ct  = patch[mid]
        p1, p99 = np.percentile(ct, [1, 99])
        ct_norm = np.clip((ct - p1) / (p99 - p1 + 1e-8), 0, 1)

        n   = len(FEATURE_NAMES)
        fig, axes = plt.subplots(2, n, figsize=(n * 3, 6))
        for i, feat in enumerate(FEATURE_NAMES):
            pred = row[f'{feat}_pred']
            gt   = row[f'{feat}_gt']
            axes[0, i].imshow(ct_norm, cmap='gray')
            axes[0, i].set_title(f'{feat}\npred {pred:.2f} / gt {gt:.2f}', fontsize=7)
            axes[0, i].axis('off')
            axes[1, i].imshow(ct_norm, cmap='gray')
            if feat in heatmaps:
                axes[1, i].imshow(heatmaps[feat][mid], cmap='jet', alpha=0.45)
            axes[1, i].axis('off')

        cand_id = row['candidate_id']
        fig.suptitle(f'{cand_id}  (TP, dist {row["match_distance_mm"]:.1f} mm)',
                     fontsize=9, fontweight='bold')
        plt.tight_layout()

        out_png = f'{DRIVE_DIR}/{cand_id}_vis.png'
        plt.savefig(out_png, dpi=150, bbox_inches='tight')
        plt.close()
        display(Image.open(out_png))
        print(f'Saved: {out_png}')